In [1]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from math import radians, cos, sin, asin, sqrt
import os

# 1. DB 및 설정 (기존 정보와 동일하게 설정하세요)
DB_CONFIG = {
    "host": "localhost",
    "user": "root",
    "pass": "1234",
    "name": "coffee_store"
}

premium_brands = ['스타벅스', '투썸플레이스', '폴바셋', '할리스', '파스쿠찌', '공차', '디저트39']
budget_brands = ['메가커피', '빽다방', '컴포즈커피', '더벤티', '메머드커피', '이디야', '던킨도너츠', '하삼동커피']

engine = create_engine(f"mysql+pymysql://{DB_CONFIG['user']}:{DB_CONFIG['pass']}@{DB_CONFIG['host']}:3306/{DB_CONFIG['name']}")

def haversine(lon1, lat1, lon2, lat2):
    lon1, lat1, lon2, lat2 = map(radians, [lon1, lat1, lon2, lat2])
    return 6371 * 2 * asin(sqrt(sin((lat2-lat1)/2)**2 + cos(lat1) * cos(lat2) * sin((lon2-lon1)/2)**2))

def get_features(station_row, stores_df):
    s_lat, s_lon = station_row['위도'], station_row['경도']
    nearby = stores_df[(stores_df['위도'].between(s_lat-0.01, s_lat+0.01)) & (stores_df['경도'].between(s_lon-0.01, s_lon+0.01))].copy()
    if nearby.empty: return pd.Series([0, 0.0, 0.0, 0])
    nearby['dist'] = nearby.apply(lambda x: haversine(s_lon, s_lat, x['경도'], x['위도']), axis=1)
    in_500m = nearby[nearby['dist'] <= 0.5]
    total = len(in_500m)
    if total == 0: return pd.Series([0, 0.0, 0.0, 0])
    return pd.Series([total, round(in_500m['브랜드명'].isin(premium_brands).sum()/total, 2), 
                      round(in_500m['브랜드명'].isin(budget_brands).sum()/total, 2), in_500m['브랜드명'].nunique()])

# --- 데이터 준비 ---
print("🔍 데이터를 불러오는 중...")
stores_df = pd.read_sql("SELECT 브랜드명, 경도, 위도 FROM coffee_chain", engine)
station_df = pd.read_csv('전체_역사정보_최종_정제_v48.csv')
stations = station_df.dropna(subset=['위도', '경도']).copy()
stations['총_승하차객수'] = stations['1월 승차이용객수'] + stations['1월 하차이용객수']

print("📊 상권 특성 추출 중 (시간이 다소 소요될 수 있습니다)...")
features = stations.apply(lambda x: get_features(x, stores_df), axis=1)
features.columns = ['브랜드_밀도', '프리미엄_비율', '가성비_비율', '브랜드_다양성']
df = pd.concat([stations, features], axis=1)

# --- 모델 정확도 개선 핵심 로직 ---
# 1. 로그 변환 (승하차객수 편차 완화)
df['총_승하차객수_log'] = np.log1p(df['총_승하차객수'])

# 2. 스케일링
target_cols = ['총_승하차객수_log', '브랜드_밀도', '프리미엄_비율', '가성비_비율', '브랜드_다양성']
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[target_cols].fillna(0))

# 3. 가중치 설정 (★ 중요: 여기서 상권 성격을 결정합니다 ★)
# [승하차객, 브랜드밀도, 프리미엄비율, 가성비비율, 브랜드다양성]
# 만약 브랜드가 없는데 초대형으로 나온다면 '브랜드_밀도'의 가중치를 대폭 높이세요.
weights = np.array([1.0, 2.5, 1.5, 1.5, 1.2]) 
X_weighted = X_scaled * weights

# --- 군집화 수행 ---
kmeans = KMeans(n_clusters=4, random_state=42, n_init=30)
df['클러스터'] = kmeans.fit_predict(X_weighted)

# --- 결과 분석 및 진단 ---
print("\n" + "="*50)
print("🧐 모델 진단 리포트")
print("="*50)

# 군집별 평균값 확인 (어떤 군집이 '초대형'인지 파악)
cluster_summary = df.groupby('클러스터')[target_cols + ['총_승하차객수']].mean()
print("\n[군집별 특성 평균]")
print(cluster_summary)

# 문제의 역 찾기 (사용자가 말한 '이상한 역'과 '강남' 비교)
target_station_name = "강남" # 비교군
problem_station_name = "" # 여기에 이상하게 나온 역 이름을 넣으세요

print(f"\n[주요 역 비교 분석]")
comparison = df[df['역명'].isin([target_station_name, problem_station_name])][['역명', '총_승하차객수', '브랜드_밀도', '클러스터']]
print(comparison)

# 팁: 특정 군집에 속한 역 명단 10개만 보기
top_cluster = cluster_summary['총_승하차객수'].idxmax()
print(f"\n[초대형 상권(군집 {top_cluster})으로 분류된 역들]")
print(df[df['클러스터'] == top_cluster]['역명'].head(15).tolist())

# 가중치 조정 가이드
print("\n💡 진단 팁:")
print("1. 브랜드가 0인데 상권이 높게 나온다면: weights의 2번째 값(브랜드_밀도)을 3.0 이상으로 올리세요.")
print("2. 강남역이 밀려난다면: weights의 1번째 값(승하차객)을 조금 더 높이세요.")

🔍 데이터를 불러오는 중...
📊 상권 특성 추출 중 (시간이 다소 소요될 수 있습니다)...

🧐 모델 진단 리포트

[군집별 특성 평균]
      총_승하차객수_log     브랜드_밀도   프리미엄_비율    가성비_비율    브랜드_다양성       총_승하차객수
클러스터                                                                     
0        7.551289   0.727273  0.255065  0.082597   0.668831   5314.818182
1        9.596696  10.834081  0.360314  0.639686   7.524664  18980.701794
2        8.710791   4.122024  0.083750  0.916250   3.380952   8359.755952
3       10.147644  26.053333  0.413467  0.586533  10.766667  41582.326667

[주요 역 비교 분석]
     역명  총_승하차객수  브랜드_밀도  클러스터
123  강남   156545    51.0     3
970  강남    39771    50.0     3

[초대형 상권(군집 3)으로 분류된 역들]
['의정부', '동대문', '종로5가', '종로3가', '종각', '시청', '서울역', '용산', '노량진', '영등포', '부천', '부평', '주안', '가산디지털단지', '안양']

💡 진단 팁:
1. 브랜드가 0인데 상권이 높게 나온다면: weights의 2번째 값(브랜드_밀도)을 3.0 이상으로 올리세요.
2. 강남역이 밀려난다면: weights의 1번째 값(승하차객)을 조금 더 높이세요.
